# UHRM prototype (Stage A, Colab): everything in one notebook

Run the cells top to bottom (Runtime > Run all works). Every section is safe to re-run; finished downloads and saved boundaries are reused.

| Section | What it does |
|---|---|
| 0 | Setup: clone the repo, mount Drive, sign in to Earth Engine |
| 1 | Access check: Earth Engine, the MODIS collection and its date coverage, Drive |
| 2 | Study areas: Mumbai and Navi Mumbai boundaries (provisional, from OpenStreetMap) |
| 3 | Model grid and city masks, with cell counts |
| 4 | First milestone: fetch one MODIS composite, filter it, map both cities |
| 5 | Phase 2: collect all datasets (MODIS LST and reflectance, ERA5-Land, WorldCover, terrain) |

**One-time prerequisites:** push the repository to GitHub so section 0 can clone it, and give it your Earth Engine Cloud project id, either by typing it into `EE_PROJECT` in the first code cell or by adding a Colab secret named `EE_PROJECT` (left sidebar, key icon, notebook access on). A project id is not a credential. Never paste passwords or tokens into a cell. If the repo is private, clone with a token kept in a Colab secret instead of the URL below.

## 0. Setup

In [ ]:
import os, subprocess, sys
REPO_URL = 'https://github.com/Starpredetor/urban-heat-risk-prediction.git'
REPO_DIR = '/content/uhrm'
if os.path.isdir(REPO_DIR):
    subprocess.run(['git', '-C', REPO_DIR, 'pull', '--ff-only'], check=True)
else:
    subprocess.run(['git', 'clone', REPO_URL, REPO_DIR], check=True)
sys.path.insert(0, REPO_DIR + '/src')
from uhrm.colab import bootstrap
project = bootstrap()

## 1. Access check
Confirms the collection id in `config/project.yaml` is valid, the band names exist, and shows the real MODIS date coverage (settles SRS open items OI-4 and OI-6).

In [ ]:
import ee
from datetime import datetime, timezone
print('Earth Engine OK:', ee.Number(1).add(1).getInfo() == 2)
col = ee.ImageCollection(project.modis.collection)
print('MOD11A2 composites available:', col.size().getInfo())
print('bands:', col.first().bandNames().getInfo())
t = col.aggregate_array('system:time_start')
first, last = t.reduce(ee.Reducer.min()).getInfo(), t.reduce(ee.Reducer.max()).getInfo()
as_date = lambda ms: datetime.fromtimestamp(ms / 1000, timezone.utc).date()
print('coverage:', as_date(first), 'to', as_date(last))

probe = project.path('_probe.txt')
probe.write_text('ok')
print('Drive write OK:', probe.read_text() == 'ok', '->', probe)
probe.unlink()

## 2. Study areas (provisional boundaries)
Fetched from OpenStreetMap once and saved to Drive with a provenance file (FR-P1.2). Later runs reuse the saved files. Stage B replaces them with authoritative boundaries verified in QGIS (FR-P1.8).

Greater Mumbai is built from the Mumbai City and Mumbai Suburban district polygons. The cell chooses the largest polygon matching each name and checks the area is plausible. If the choice looks wrong, set `MUMBAI_IDS` / `NAVI_MUMBAI_IDS` from the printed candidate list and set `REFRESH_BOUNDARIES = True`. Which Navi Mumbai polygon is right is open item OI-1.

In [ ]:
import json
from uhrm import boundaries_osm as osm, regions

REFRESH_BOUNDARIES = False   # True: search OpenStreetMap again and overwrite the saved files
MUMBAI_IDS = None            # optional manual choice, e.g. [1234567, 7654321]
NAVI_MUMBAI_IDS = None       # optional manual choice, e.g. [13180880]

paths = {a.slug: project.boundary_path(a) for a in project.study_areas}
if REFRESH_BOUNDARIES or not all(p.exists() for p in paths.values()):
    navi = osm.search('Navi Mumbai')
    mumbai = osm.search('Mumbai City district') + osm.search('Mumbai Suburban district')
    print('--- Navi Mumbai candidates'); osm.show(navi)
    print('--- Mumbai candidates'); osm.show(mumbai)
    navi_pick = osm.pick(navi, NAVI_MUMBAI_IDS) if NAVI_MUMBAI_IDS else osm.pick_largest_by_prefix(navi, ['Navi Mumbai'])
    mumbai_pick = osm.pick(mumbai, MUMBAI_IDS) if MUMBAI_IDS else osm.pick_largest_by_prefix(mumbai, ['Mumbai City', 'Mumbai Suburban'])
    osm.save_boundary(navi_pick, paths['navi_mumbai'], query='Navi Mumbai')
    osm.save_boundary(mumbai_pick, paths['mumbai'], query='Mumbai City + Mumbai Suburban districts')
else:
    print('Using saved boundaries (set REFRESH_BOUNDARIES = True to fetch again).')

for slug, p in paths.items():
    meta = json.loads(p.with_suffix('.meta.json').read_text(encoding='utf-8'))
    print(f"{slug}: ~{meta['approx_area_km2']:.0f} km2 from", [s['display_name'][:40] for s in meta['sources']])
    # A building or a point has an area near 0; this catches a wrong pick early.
    assert 20 < meta['approx_area_km2'] < 3000, f'{slug}: area looks wrong, choose ids by hand (see the candidate list)'

## 3. Model grid and city masks
A buffered 1 km model domain around both cities (FR-P1.4); the city polygons stay the masks for analysis and metrics. Cell counts show how much data each city really has.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

geoms = {a.slug: regions.load_boundary(project.boundary_path(a), project.grid.crs) for a in project.study_areas}
grid = regions.Grid.covering(list(geoms.values()), res=project.grid.model_resolution_m,
                             buffer_m=project.grid.domain_buffer_km * 1000, crs=project.grid.crs)
masks = {slug: grid.cell_mask(g, project.grid.min_cell_overlap) for slug, g in geoms.items()}
for k, v in regions.cell_report(grid, masks).items():
    print(f'{k}: {v}')
regions.save_grid(grid, project.path('grid.json'))

canvas = np.zeros(grid.shape)
for i, m in enumerate(masks.values(), start=1):
    canvas[m] = i
x0, y0, x1, y1 = grid.bounds
fig, ax = plt.subplots(figsize=(7, 8))
ax.imshow(canvas, extent=(x0, x1, y0, y1), cmap='viridis')
for g in geoms.values():
    for poly in getattr(g, 'geoms', [g]):
        ax.plot(*poly.exterior.xy, 'w-', lw=0.8)
ax.set_title('Model domain: 1 km cells assigned to each city')
plt.show()

**Check:** both outlines sit where the cities are; `overlap_between_cities` is 0; the grid is large enough for 64 x 64 patches (OI-3). Note the cell counts: small counts limit the statistical power of per-city results.

## 4. First milestone: one MODIS LST map per city
Fetches a single MOD11A2 composite, saves it as a raw GeoTIFF with a catalogue row (never modified, FR-P2.6), then quality-filters and scales it. Do this before downloading 25 years (FR-P2.9).

This is the first real call to Earth Engine's `computePixels`, so an error here is useful feedback about the backend.

In [ ]:
from datetime import timedelta
from uhrm import acquire, modis
from uhrm.backends.earthengine import EarthEngineModis

backend = EarthEngineModis(project.modis.collection)
start = project.modis.first_milestone_date
dates = backend.list_composites(start, start + timedelta(days=40))
assert dates, f'no composites found after {start}; check the collection id and date range'
day = dates[0]
print('composites found:', dates, '| using', day)

out_dir = project.path('raw', 'modis')
print(acquire.acquire_modis(backend, grid, [day], out_dir, project.path('catalog.csv')))

In [ ]:
import rasterio
with rasterio.open(acquire.composite_path(out_dir, day)) as src:
    dn, qc = src.read(1), src.read(2)
lst = modis.process_composite(dn, qc, project.modis.qc)
print(f'whole domain: {np.isfinite(lst).mean():.0%} of cells valid')
for slug, m in masks.items():
    v = lst[m]
    ok = np.isfinite(v)
    if ok.any():
        print(f'{slug}: {m.sum()} cells, {ok.mean():.0%} valid, LST {v[ok].min():.1f} to {v[ok].max():.1f} C, mean {v[ok].mean():.1f} C')
    else:
        print(f'{slug}: {m.sum()} cells, none valid on this date')

fig, axes = plt.subplots(1, 2, figsize=(13, 7))
im = axes[0].imshow(lst, extent=(x0, x1, y0, y1), cmap='inferno')
axes[0].set_title(f'MOD11A2 daytime LST (C), composite starting {day}')
fig.colorbar(im, ax=axes[0], shrink=0.7)
axes[1].imshow(np.isfinite(lst), extent=(x0, x1, y0, y1), cmap='gray')
axes[1].set_title('valid cells (white) after quality filter')
for ax in axes:
    for g in geoms.values():
        for poly in getattr(g, 'geoms', [g]):
            ax.plot(*poly.exterior.xy, 'c-', lw=0.8)
plt.tight_layout()
plt.show()

### Milestone checklist
- April daytime LST is plausible for land (roughly 30-50 C); the sea is cooler than the land.
- The coastline and both outlines line up with the temperature field.
- Rejected cells are where cloud or coast would be expected, not a random pattern.
- Both cities show some valid cells. If one has none on this date, try another composite before concluding anything.
- Download the saved GeoTIFF from Drive and open it in QGIS to confirm it lines up with the boundary. That is the final check for this milestone.

## 5. Phase 2: collect all datasets
Fetches, for the model domain, every Stage A dataset and saves each as a raw GeoTIFF on Drive with a catalogue row (checksum included):

| Dataset | Files | Grid |
|---|---|---|
| MODIS MOD11A2 daytime LST + QC | one per 8-day composite | 1 km |
| MODIS MOD09A1 surface reflectance + state QA | one per 8-day composite | 500 m (aggregated later) |
| ERA5-Land temperature, dewpoint, wind, skin temperature | one per 8-day window, averaged over the Terra overpass hours | 1 km grid (about 9 km native) |
| ESA WorldCover class fractions (2020, 2021) | one per release | 1 km |
| SRTM elevation mean and relief | one | 1 km |

**Time it first.** Run with `YEARS = [2024]` (46 composites per series) and note how long it takes; then switch to the full range. The cell is safe to re-run after any disconnect: finished files are skipped, failed fetches retry with backoff and are listed, and a coverage line per series shows how many composites are on disk.

Landsat 8/9 annual composites are not in this section yet; they feed the Phase 5 detail maps and need their own per-city 30 m grids.

In [ ]:
import time
from uhrm import phase2

YEARS = [2024]                 # first run: one year, to time it and check the files
# YEARS = list(range(2001, 2026))   # then the full series (finished files are skipped)
WHICH = phase2.ALL             # or one at a time, e.g. ('modis_lst',) or ('era5_land',)

t0 = time.time()
result = phase2.run_phase2(project, grid, YEARS, phase2.earthengine_backends(project), which=WHICH)
print(f'finished in {time.time() - t0:.0f} s')

In [ ]:
import csv, collections
import rasterio

rows = list(csv.DictReader(open(project.path('catalog.csv'), encoding='utf-8')))
print('catalogue rows by dataset:', dict(collections.Counter(r['dataset'] for r in rows)))

def peek(path):
    print(path.name)
    with rasterio.open(path) as src:
        for i in range(1, src.count + 1):
            a = src.read(i).astype('float64')
            print(f'  band {i} {src.descriptions[i - 1]!r}: min {np.nanmin(a):.4g}  max {np.nanmax(a):.4g}  mean {np.nanmean(a):.4g}')

raw = project.path('raw')
for pattern in ['modis_reflectance/MOD09A1_*.tif', 'era5/ERA5L_*.tif', 'worldcover/*.tif', 'terrain/*.tif']:
    files = sorted(raw.glob(pattern))
    if files:
        peek(files[len(files) // 2])        # a file from the middle of the year

wc = sorted(raw.glob('worldcover/*2021*.tif'))
if wc:
    with rasterio.open(wc[0]) as src:
        total = src.read().sum(axis=0)
    print('WorldCover fraction sums per cell: min', total.min().round(3), 'max', total.max().round(3))

### Phase 2 checks
- ERA5-Land `temperature_2m` is around 295-310 K in March-May; the wind and dewpoint values look sensible.
- WorldCover fractions sum to about 1 in every cell, and built-up (`c50`) is high in the city centres.
- Terrain: the sea is near 0 m, and the hills and the Sahyadri edge show up as higher `elevation_mean` and `elevation_stdDev`.
- Reflectance: the `valid` band is mostly 1; near-infrared (`sur_refl_b02`) exceeds red (`sur_refl_b01`) over vegetation. Large negative values would be fill values that Earth Engine did not mask (Phase 3 screens them).
- Every series should show 46 composites per year in the coverage lines above, except the last months of the final year for ERA5-Land, which lags real time.